In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# 셀 1 — 설치
!pip install streamlit pyngrok deep-translator -q

!pip install ftfy regex tqdm -q
!pip install git+https://github.com/openai/CLIP.git -q
!pip install faiss-cpu -q

  Preparing metadata (setup.py) ... done


In [23]:
from pyngrok import ngrok
ngrok.kill()
print("✅ 기존 터널 종료 완료!")

✅ 기존 터널 종료 완료!


In [24]:
# 필수 코드
%cd /content/drive/MyDrive/SafeSight

/content/drive/MyDrive/SafeSight


In [8]:
!mkdir -p app

In [25]:
%%writefile requirements.txt
streamlit
pillow
torch
torchvision
scikit-learn
open_clip_torch
faiss-cpu
pandas
numpy
deep_translator

Overwriting requirements.txt


In [51]:
%%writefile app/streamlit_ui.py
import streamlit as st
import clip
import torch
import faiss
import numpy as np
import pandas as pd
from PIL import Image
import os
from deep_translator import GoogleTranslator
from ultralytics import YOLO

@st.cache_resource
def load_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, preprocess = clip.load("ViT-B/32", device=device)
    yolo_model = YOLO("yolov8n.pt")
    return model, preprocess, yolo_model, device

@st.cache_resource
def load_db():
    index = faiss.read_index(
        '/content/drive/MyDrive/SafeSight/data/embeddings/image_index.faiss'
    )
    index_map = pd.read_csv(
        '/content/drive/MyDrive/SafeSight/data/embeddings/index_map.csv'
    )
    metadata = pd.read_csv(
        '/content/drive/MyDrive/SafeSight/data/raw/metadata.csv'
    )
    return index, index_map, metadata

model, preprocess, yolo_model, device = load_model()
index, index_map, metadata = load_db()
IMG_DIR = '/content/drive/MyDrive/SafeSight/data/raw/images'

CAT_KEYWORDS = ['고양이', '묘', '코리안숏헤어', '페르시안', '러시안블루', '스핑크스', '랙돌', '샴']

def search(query=None, pil_img=None, k=5, animal_type=None):
    with torch.no_grad():
        if pil_img is not None:
            image_input = preprocess(pil_img).unsqueeze(0).to(device)
            emb = model.encode_image(image_input)
        else:
            try:
                query_en = GoogleTranslator(source='ko', target='en').translate(query)
            except:
                query_en = query
            text = clip.tokenize([query_en]).to(device)
            emb = model.encode_text(text)

        emb = emb / emb.norm(dim=-1, keepdim=True)
        emb_np = emb.cpu().numpy().astype('float32')

    total = index.ntotal
    similarities, indices = index.search(emb_np, total)

    results = []
    for sim, idx in zip(similarities[0], indices[0]):
        desertion_no = index_map.iloc[idx]['desertionNo']
        meta_row = metadata[metadata['desertionNo'] == int(desertion_no)]
        if len(meta_row) == 0:
            continue
        meta = meta_row.iloc[0]

        if animal_type == "🐶 강아지":
            if any(cat in str(meta['kindNm']) for cat in CAT_KEYWORDS):
                continue
        if animal_type == "🐱 고양이":
            if not any(cat in str(meta['kindNm']) for cat in CAT_KEYWORDS):
                continue

        results.append({
            "id": str(desertion_no),
            "similarity": round(float(sim) * 100, 1),
            "kindNm": meta['kindNm'],
            "colorCd": meta['colorCd'],
            "specialMark": meta['specialMark'],
            "happenPlace": meta['happenPlace'],
            "orgNm": meta['orgNm'],
            "img_path": f"{IMG_DIR}/{desertion_no}.jpg"
        })

        if len(results) >= k:
            break

    return results

if "page" not in st.session_state:
    st.session_state.page = "main"
if "query" not in st.session_state:
    st.session_state.query = ""
if "animal_type" not in st.session_state:
    st.session_state.animal_type = ""
if "results" not in st.session_state:
    st.session_state.results = []
if "cropped_img_path" not in st.session_state:
    st.session_state.cropped_img_path = None

def main_page():
    st.title("🐾 SafeSight")
    st.caption("놓치지 않는 시선, 연결되는 안전")
    st.divider()

    animal_type = st.radio(
        "동물 종류",
        ["🐶 강아지", "🐱 고양이"],
        horizontal=True
    )

    st.divider()
    st.subheader("🔍 인상착의 입력")
    query = st.text_input(
        "찾는 반려동물 특징을 입력하세요",
        placeholder="예: 흰색 말티즈 빨간 목줄 착용"
    )

    st.subheader("📸 사진 업로드")
    image = st.file_uploader(
        "또는 사진으로 검색",
        type=["jpg", "jpeg", "png"]
    )
    if image:
        st.image(image, caption="업로드된 사진", width=200)

    st.divider()

    if st.button("🔍 탐색 시작", use_container_width=True):
        if not query and not image:
            st.warning("⚠️ 특징을 입력하거나 사진을 업로드해주세요!")
        else:
            with st.spinner("🔍 탐색 중..."):
                if image is not None:
                    input_image = Image.open(image)
                    yolo_results = yolo_model(input_image)
                    cropped_image = None

                    for res in yolo_results:
                        for box in res.boxes:
                            cls_id = int(box.cls[0])
                            if cls_id in [15, 16]:
                                x1, y1, x2, y2 = map(int, box.xyxy[0])
                                cropped_image = input_image.crop((x1, y1, x2, y2))
                                break

                    if cropped_image is None:
                        cropped_image = input_image

                    results = search(pil_img=cropped_image, animal_type=animal_type)

                    # 경로 안전 확보 후 크롭 이미지 임시 저장
                    os.makedirs("app", exist_ok=True)
                    cropped_image.save("app/temp_crop.jpg")
                    st.session_state.cropped_img_path = "app/temp_crop.jpg"
                else:
                    results = search(query=query, animal_type=animal_type)
                    st.session_state.cropped_img_path = None

            st.session_state.page = "result"
            st.session_state.query = query if query else "업로드된 이미지 기준 검색"
            st.session_state.animal_type = animal_type
            st.session_state.results = results
            st.rerun()

def result_page():
    if st.button("← 뒤로가기"):
        st.session_state.page = "main"
        st.rerun()

    st.title("검색 결과")

    col1, col2 = st.columns([4, 1])
    with col1:
        st.caption(f'🔍 "{st.session_state.query}"')
        st.caption(f'🐾 {st.session_state.animal_type}')
    with col2:
        st.metric("검색 건수", f"{len(st.session_state.results)}건")

    if st.session_state.cropped_img_path and os.path.exists(st.session_state.cropped_img_path):
        st.image(st.session_state.cropped_img_path, caption="YOLOv8 탐지 및 크롭 영역", width=150)

    st.divider()

    for result in st.session_state.results:
        with st.container(border=True):
            col1, col2 = st.columns([3, 1])

            with col1:
                try:
                    st.image(result['img_path'], width=100)
                except:
                    st.write("🐾")

                st.markdown(f"**공고 #{result['id']}**")
                st.caption(f"📍 {result['happenPlace']} ({result['orgNm']})")

                tag_html = " ".join([
                    f'<span style="background:#e6f1fb;padding:2px 8px;border-radius:10px;font-size:12px;margin-right:4px">{result["kindNm"]}</span>',
                    f'<span style="background:#eaf3de;padding:2px 8px;border-radius:10px;font-size:12px;margin-right:4px">{result["colorCd"]}</span>',
                ])
                st.markdown(tag_html, unsafe_allow_html=True)
                st.caption(f"특징: {result['specialMark']}")

            with col2:
                score = result["similarity"]
                color = "#185fa5" if score >= 50 else "#854f0b"
                # 🌟 [오타 완벽 수정 완료] 엉뚱한 인자 조각 제거
                st.markdown(
                    f'<div style="text-align:right;font-size:24px;font-weight:bold;color:{color}">{score}%</div>',
                    unsafe_allow_html=True
                )
                st.progress(score / 100)

if st.session_state.page == "main":
    main_page()
elif st.session_state.page == "result":
    result_page()

Overwriting app/streamlit_ui.py


In [33]:
!pip install ultralytics deep-translator

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.7 MB/s eta 0:00:00


In [29]:
import subprocess, threading, time

def run():
    subprocess.Popen([
        "streamlit", "run", "app/streamlit_ui.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

threading.Thread(target=run).start()
time.sleep(8)  # 8초 대기 (CLIP 로드 시간 필요)
print("✅ Streamlit 실행 완료!")

✅ Streamlit 실행 완료!


In [12]:
from pyngrok import ngrok
ngrok.kill()

In [55]:
# 셀 4 — ngrok 연결 (Streamlit 실행 후에!)
from pyngrok import ngrok
ngrok.set_auth_token("3Dt9gJxV8SezvAzkGzhSEojZgle_3xFDgQ78jCtLB6nahFvy4")

public_url = ngrok.connect(8501)
print(f"✅ 접속 URL: {public_url}")

✅ 접속 URL: NgrokTunnel: "https://charm-guy-degrease.ngrok-free.dev" -> "http://localhost:8501"


In [57]:
!streamlit run app/streamlit_ui.py &



2026-06-08 09:06:35.346 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.192.104.198:8501

  Stopping...
